# IMPORTS

In [0]:
import requests
from pyspark.sql import SparkSession
from pyspark.sql.functions import regexp_extract_all, regexp_replace,col, lit, size,  explode,split, element_at,col,udf,length
from pyspark.sql.types import StringType
import json
import datetime

# Extract

In [0]:
spark = SparkSession.getActiveSession()
if spark is None:
    spark = SparkSession.builder.getOrCreate()

## Reddit

In [0]:
reddit_post = spark.read.csv(
    "/Volumes/market-mood/raw/news/reddit_wsb.csv",
    header=True,
    inferSchema=True
)
reddit_post.printSchema()

In [0]:
reddit_post = reddit_post.withColumn(
    "tickers",
    regexp_extract_all(col("body"), lit(r'\$([A-Z]{1,5})'), 1)
).select("body", "tickers")

In [0]:
reddit_post.filter(size('tickers') == 0).count()/reddit_post.count()*100

In [0]:
reddit_post.filter(reddit_post['body'].isNull()).count()

## Twitter


In [0]:
tweet_df = spark.read.csv(
    "/Volumes/market-mood/raw/social/stock.csv",
    header=True,
    inferSchema=True
)
tweet_df.printSchema()

In [0]:
json.loads(tweet_df.filter(~tweet_df['financial_info'].isNull()).select('financial_info').first()[0].replace("'",'"'))[0]['ticker']


# Transform

In [0]:
SP500_UNIVERSE = [
    "AAPL", "MSFT", "GOOGL", "AMZN", "NVDA", "META", "TSLA", "BRK-B",
    "JPM", "JNJ", "V", "PG", "UNH", "HD", "MA", "DIS", "PYPL", "BAC",
    "NFLX", "ADBE"
]

## Reddit

In [0]:
reddit_post = reddit_post.withColumn("ticker",explode(col('tickers')))
reddit_post.printSchema()

In [0]:
reddit_post = reddit_post.withColumn("ticker",regexp_replace(reddit_post['ticker'],r'^\$',''))

In [0]:
reddit_post.filter(reddit_post['ticker'].isin(SP500_UNIVERSE)).count()
reddit_post.groupBy("ticker").count().orderBy("count", ascending=False).show(20)

## Twitter

In [0]:
def extract_ticker(x):
    try:
        return json.loads(x.replace("'", '"'))[0]['ticker']
    except:
        return None

In [0]:
tweet_df = tweet_df.filter((~tweet_df['financial_info'].isNull())&(length(col('financial_info')) > 2))
extraer_ticker = udf(extract_ticker,StringType())#user defined fucntion para aplicar sobre los valores de una columna
tweet_df = tweet_df.withColumn('ticker',extraer_ticker(col('financial_info')))

In [0]:
tweet_df = tweet_df.withColumn("ticker",regexp_replace(tweet_df['ticker'],r'^\$',''))

In [0]:
# Ver algunos valores de financial_info para entender el problema
# tweet_df.select('ticker').show(5)
tweet_df = tweet_df.filter(~tweet_df['ticker'].isNull())

In [0]:
tweet_df = tweet_df.filter(col('ticker').isNotNull())
print(f"Filas después de filtrar nulos: {tweet_df.count()}")

In [0]:
tweet_df = tweet_df.filter(tweet_df['ticker'].isin(SP500_UNIVERSE))

In [0]:
tweet_df = tweet_df.select('timestamp', 'description', 'url', 'financial_info', 'sentiment', 'ticker')


In [0]:
# date_str = udf(lambda x: datetime.datetime.strptime(x, "%a %b %d %H:%M:%S %Z %Y"))
tweet_df = tweet_df.withColumn('date',tweet_df['timestamp'].try_cast('date'))


In [0]:
tweet_df  = tweet_df.select("date", "description", "sentiment", "ticker")
tweet_df.printSchema()

# Load

In [0]:
path = '/Volumes/market-mood/processed/sentiment_scores'
tweet_df.write.format("delta").mode("overwrite").save(path)